<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH12/CH12_NB01_vllm_attention_free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Rearchitecting LLMs
## Structural techniques for efficient models

### Chapter 12: Case study — Replacing one agent call with a specialized SLM in production — NB01: Serving a Model with Attention-Free Blocks in vLLM

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: GPU, vLLM 0.30.0, float16

Models:
- `meta-llama/Llama-3.2-3B` — unmodified control model
- `oopere/llama-3.2-3b-attn-drop-6` — model with attention removed from 6 of its 28 blocks

_____
This notebook checks whether vLLM can serve a structurally pruned model whose attention sub-block has been removed from six decoder blocks. It compares the control with the pruned checkpoint, first loading it the standard way and then with a small custom vLLM model class. The comparison focuses on successful generation, allocated KV cache, and batch-1 latency; it does not evaluate model quality.

The control model is gated: accept its license on the [Llama 3.2 3B model page](https://huggingface.co/meta-llama/Llama-3.2-3B) and authenticate with a Hugging Face token. Each vLLM load runs in a separate process so failed loads cannot leave the notebook's GPU in a bad state.

## 1. Setup

In [ ]:
!pip -q install vllm==0.30.0 huggingface_hub

Installing vLLM can replace the torch version that Colab ships. This notebook never imports torch or vLLM in the kernel itself, only in subprocesses, so no restart is needed. The next cell prints the versions a fresh process sees.

In [ ]:
import json, subprocess, sys
from pathlib import Path

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # subprocesses inherit it

WORK_DIR = Path("ch12_vllm")
(WORK_DIR / "logs").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "attn_drop_plugin" / "attn_drop_vllm").mkdir(parents=True, exist_ok=True)

VERSIONS = r'''
import json, torch, transformers, vllm
print(json.dumps({"vllm": vllm.__version__, "torch": torch.__version__,
                  "transformers": transformers.__version__, "cuda": torch.version.cuda,
                  "gpu": torch.cuda.get_device_name(0)}, indent=2))
'''
print(subprocess.run([sys.executable, "-c", VERSIONS], capture_output=True, text=True).stdout)

{
  "vllm": "0.30.0",
  "torch": "2.13.0+cu130",
  "transformers": "5.18.0",
  "cuda": "13.0",
  "gpu": "Tesla T4"
}



The control model, `meta-llama/Llama-3.2-3B`, is gated. Accept the license on its [model page](https://huggingface.co/meta-llama/Llama-3.2-3B) and log in with a Hugging Face token.

In [ ]:
TARGET_ID = "oopere/llama-3.2-3b-attn-drop-6"
CONTROL_ID = "meta-llama/Llama-3.2-3B"

# One configuration for every load, so the comparison is fair.
# float16 rather than bfloat16, so the same notebook also runs on a T4.
LLM_KWARGS = dict(dtype="half", max_model_len=2048, gpu_memory_utilization=0.85,
                  enforce_eager=True, trust_remote_code=True)

KV_HEADS, HEAD_DIM, BYTES_FP16 = 8, 128, 2
PER_BLOCK = 2 * KV_HEADS * HEAD_DIM * BYTES_FP16       # K and V for one attention block
FORMULA = {"28 attention blocks": PER_BLOCK * 28, "22 attention blocks": PER_BLOCK * 22}
print(FORMULA)

{'28 attention blocks': 114688, '22 attention blocks': 90112}


## 2. What the checkpoint contains

In [ ]:
from huggingface_hub import hf_hub_download

TARGET_ID = "oopere/llama-3.2-3b-attn-drop-6"

target_config = json.load(
    open(hf_hub_download(TARGET_ID, "config.json")))  #A
DROPPED = target_config["dropped_attn_layers"]
print("architectures      :",
      target_config["architectures"])
print("num_hidden_layers  :",
      target_config["num_hidden_layers"])
print("dropped_attn_layers:", DROPPED)

weight_map = json.load(open(hf_hub_download(
    TARGET_ID, "model.safetensors.index.json"
)))["weight_map"]                                    #B
for layer in (17, DROPPED[0]):
    names = sorted(
        k.split(f"layers.{layer}.")[1]
        for k in weight_map
        if f".layers.{layer}." in k)
    print(f"\nblock {layer}:", names)

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

architectures      : ['LlamaForCausalLM']
num_hidden_layers  : 28
dropped_attn_layers: [18, 20, 21, 22, 23, 24]


model.safetensors.index.json:   0%|          | 0.00/18.5k [00:00<?, ?B/s]


block 17: ['input_layernorm.weight', 'mlp.down_proj.weight', 'mlp.gate_proj.weight', 'mlp.up_proj.weight', 'post_attention_layernorm.weight', 'self_attn.k_proj.weight', 'self_attn.o_proj.weight', 'self_attn.q_proj.weight', 'self_attn.v_proj.weight']

block 18: ['mlp.down_proj.weight', 'mlp.gate_proj.weight', 'mlp.up_proj.weight', 'post_attention_layernorm.weight']


The attention-free blocks have no `q/k/v/o_proj` and no `input_layernorm`, the norm that feeds attention. Only `post_attention_layernorm` and the MLP remain. The config still declares the standard `LlamaForCausalLM` architecture. The repo also ships a custom Transformers class through `auto_map`, which Transformers uses with `trust_remote_code=True`.

## 3. The vLLM probe

One script, run in a fresh process for every model. It loads the model with `LLM_KWARGS` and, if the load succeeds, it runs two tests:

- **Free generation:** greedy, 30 tokens.
- **Batch-1 latency:** median of 10 runs after a warm-up.

vLLM writes the KV cache it allocated to its log. `run_vllm` prints those lines, the result line and, if the load fails, the first error.

In [ ]:
%%writefile ch12_vllm/vllm_probe.py
import ast, statistics, sys, time, traceback

model_id, kwargs = sys.argv[1], ast.literal_eval(sys.argv[2])

PROMPT = "The three most visited cities in Europe are"
N_TOKENS = 30

try:
    from vllm import LLM, SamplingParams

    llm = LLM(model=model_id, **kwargs)

    # ignore_eos keeps every run at exactly N_TOKENS, so runs are comparable
    free = SamplingParams(temperature=0.0, max_tokens=N_TOKENS, ignore_eos=True)
    text = llm.generate([PROMPT], free)[0].outputs[0].text   # also the warm-up

    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        llm.generate([PROMPT], free, use_tqdm=False)
        times.append((time.perf_counter() - t0) * 1000)
    print(f"RESULT ms_per_token={statistics.median(times) / N_TOKENS:.2f} text={text!r}", flush=True)
except Exception:
    traceback.print_exc()


Writing ch12_vllm/vllm_probe.py


In [ ]:
def run_vllm(model_id, name, extra_kwargs=None):
    # Runs the probe in its own process; the full stdout/stderr goes to a log file
    log_path = WORK_DIR / "logs" / f"{name}.log"
    kwargs = {**LLM_KWARGS, **(extra_kwargs or {})}
    with open(log_path, "w") as log:
        subprocess.run([sys.executable, str(WORK_DIR / "vllm_probe.py"), model_id, repr(kwargs)],
                       stdout=log, stderr=subprocess.STDOUT)
    log_lines = log_path.read_text(errors="replace").splitlines()

    for l in log_lines:
        if "Available KV cache memory" in l or "GPU KV cache size" in l or "RESULT " in l:
            print(l)
    if not any("RESULT " in l for l in log_lines):
        # vLLM's last error ("Engine core initialization failed") is a wrapper;
        # the first error in the log is the real cause
        print(next((l for l in log_lines
                    if "Error:" in l and "Engine core initialization failed" not in l), None))
    print(f"(log: {log_path})")


## 4. Control: an unmodified Llama-3.2-3B

In [ ]:
run_vllm(CONTROL_ID, "control")

(EngineCore pid=4098) INFO 10-04 18:24:45 [gpu_worker.py:640] Available KV cache memory: 5.63 GiB
(EngineCore pid=4098) INFO 10-04 18:24:45 [kv_cache_utils.py:2395] GPU KV cache size: 52,720 tokens, Maximum concurrency for 2,048 tokens per request: 25.74x
RESULT ms_per_token=33.55 text=' London, Paris and Rome. These cities are the most popular tourist destinations in Europe. The most visited city in Europe is London. London is the capital'
(log: ch12_vllm/logs/control.log)


To get the bytes per token, convert the KV cache memory to bytes (GiB × 2³⁰) and divide it by the number of tokens. The result should be close to the 114,688 bytes/token of the formula for 28 attention blocks. The small gap comes from vLLM rounding the GiB figure to two decimals.


## 5. The pruned model, the standard way

In [ ]:
run_vllm(TARGET_ID, "attn_drop_standard")

(EngineCore pid=4788) ERROR 10-04 18:26:16 [core.py:1366] ValueError: Following weights were not initialized from checkpoint: {'model.layers.21.input_layernorm.weight', 'model.layers.18.input_layernorm.weight', 'model.layers.24.input_layernorm.weight', 'model.layers.23.input_layernorm.weight', 'model.layers.22.input_layernorm.weight', 'model.layers.20.input_layernorm.weight'}
(log: ch12_vllm/logs/attn_drop_standard.log)


The load fails after reading the weights:

```
ValueError: Following weights were not initialized from checkpoint:
{'model.layers.18.input_layernorm.weight', 'model.layers.20.input_layernorm.weight', ...}
```

vLLM does not run the repo's custom Transformers class. It reads `architectures: ["LlamaForCausalLM"]` from the config and builds its own optimized Llama, which assumes that all 28 blocks are identical: every block has attention and the norm that feeds it. The checkpoint doesn't have those weights for the 6 pruned blocks, so the load stops. `trust_remote_code=True` doesn't change this. The checkpoint is fine; the engine's model definition is what is missing.

## 6. The pruned model with a custom vLLM model class

vLLM lets you register your own architectures through a plugin. A plugin is a small Python package with a `vllm.general_plugins` entry point, which vLLM loads in every process, including the engine core. The class does not need to rewrite Llama. vLLM's `LlamaForCausalLM` accepts a `layer_type` argument, so we only need a decoder layer that, for the blocks listed in `dropped_attn_layers`:

- builds only `post_attention_layernorm` and the MLP, which matches what the checkpoint stores;
- computes `x + mlp(norm(x))`, the same as the repo's Transformers class;
- registers no `Attention` layer. vLLM allocates KV cache per attention layer, so these blocks get none.

In [ ]:
%%writefile ch12_vllm/attn_drop_plugin/attn_drop_vllm/model.py
from torch import nn

from vllm.model_executor.layers.layernorm import RMSNorm
from vllm.model_executor.models.llama import (
    LlamaAttention, LlamaDecoderLayer, LlamaForCausalLM, LlamaMLP)
from vllm.model_executor.models.utils import extract_layer_index


class AttnFreeAwareDecoderLayer(LlamaDecoderLayer):
    def __init__(self, vllm_config, prefix="", config=None, attn_layer_type=LlamaAttention):
        config = config or vllm_config.model_config.hf_config
        dropped = set(getattr(config, "dropped_attn_layers", None) or [])
        if extract_layer_index(prefix) not in dropped:
            # A normal block: vLLM's own implementation, unchanged
            super().__init__(vllm_config=vllm_config, prefix=prefix, config=config,
                             attn_layer_type=attn_layer_type)
            self.has_attention = True
            return

        # An attention-free block: only the MLP and its norm, no Attention layer
        nn.Module.__init__(self)
        self.has_attention = False
        self.hidden_size = config.hidden_size
        self.mlp = LlamaMLP(
            hidden_size=config.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=config.hidden_act,
            quant_config=self.get_quant_config(vllm_config),
            bias=getattr(config, "mlp_bias", False),
            prefix=f"{prefix}.mlp",
        )
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, positions, hidden_states, residual):
        if self.has_attention:
            return super().forward(positions, hidden_states, residual)
        # vLLM carries the residual stream as (pending output, residual). The fused
        # add + norm folds the previous block's output in, so this is x + mlp(norm(x))
        if residual is None:
            residual = hidden_states
            hidden_states = self.post_attention_layernorm(hidden_states)
        else:
            hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class AttnDropLlamaForCausalLM(LlamaForCausalLM):
    def __init__(self, *, vllm_config, prefix=""):
        if not getattr(vllm_config.model_config.hf_config, "dropped_attn_layers", None):
            raise ValueError("AttnDropLlamaForCausalLM needs config.dropped_attn_layers")
        super().__init__(vllm_config=vllm_config, prefix=prefix,
                         layer_type=AttnFreeAwareDecoderLayer)

Writing ch12_vllm/attn_drop_plugin/attn_drop_vllm/model.py


In [ ]:
%%writefile ch12_vllm/attn_drop_plugin/attn_drop_vllm/__init__.py
def register():
    from vllm import ModelRegistry
    # A "module:Class" string keeps the import lazy, so loading the plugin doesn't touch CUDA
    if "AttnDropLlamaForCausalLM" not in ModelRegistry.get_supported_archs():
        ModelRegistry.register_model("AttnDropLlamaForCausalLM",
                                     "attn_drop_vllm.model:AttnDropLlamaForCausalLM")

Writing ch12_vllm/attn_drop_plugin/attn_drop_vllm/__init__.py


In [ ]:
%%writefile ch12_vllm/attn_drop_plugin/pyproject.toml
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "attn-drop-vllm"
version = "0.1.0"

[project.entry-points."vllm.general_plugins"]
attn_drop = "attn_drop_vllm:register"

[tool.setuptools]
packages = ["attn_drop_vllm"]

Writing ch12_vllm/attn_drop_plugin/pyproject.toml


In [ ]:
%pip -q install --no-deps --force-reinstall ./ch12_vllm/attn_drop_plugin

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


The checkpoint stays as it is. `hf_overrides` changes the config only in memory: it points vLLM to the new architecture name and passes the list of attention-free blocks.

In [ ]:
custom = run_vllm(TARGET_ID, "attn_drop_custom", extra_kwargs={
    "hf_overrides": {"architectures": ["AttnDropLlamaForCausalLM"], "dropped_attn_layers": DROPPED}})

(EngineCore pid=5346) INFO 10-04 18:27:10 [gpu_worker.py:640] Available KV cache memory: 5.91 GiB
(EngineCore pid=5346) INFO 10-04 18:27:10 [kv_cache_utils.py:2395] GPU KV cache size: 70,464 tokens, Maximum concurrency for 2,048 tokens per request: 34.41x
RESULT ms_per_token=27.23 text=' London, Paris and Paris. The capital of France is the most visited city in Europe. The city is the capital of France and the capital of the'
(log: ch12_vllm/logs/attn_drop_custom.log)


Same division as for the control: the result should be close to the 90,112 bytes/token of the formula for 22 attention blocks.


## 7. What the numbers say

Results from this run (NVIDIA T4, vLLM 0.30.0, torch 2.13.0, transformers 5.17.0, float16, `enforce_eager=True`):

| | Control (28 attention blocks) | attn-drop-6, custom class (22) |
|---|---|---|
| Available KV cache memory | 5.63 GiB | 5.91 GiB |
| GPU KV cache capacity | 52,720 tokens | 70,464 tokens |
| Bytes per token (from rounded log memory) | ~114,666 (formula: 114,688) | ~90,058 (formula: 90,112) |
| Batch-1 median latency | 33.55 ms/token | 27.23 ms/token |

- **The cache-memory figure is not the model's total memory requirement.** In this run, vLLM reported 0.28 GiB more KV-cache memory available for the pruned model (5.91 vs. 5.63 GiB). These logs do not independently measure the model-weight difference or isolate what caused the change.
- **The per-token KV-cache saving matches the layer-count formula.** The formula gives 90,112 bytes/token for 22 attention blocks versus 114,688 for 28, about 21.4% less. The reported cache capacity increases from 52,720 to 70,464 tokens, about 33.7% in this run.
- **Batch-1 latency was about 18.8% lower in this run**, from 33.55 to 27.23 ms/token. This measurement alone does not establish the cause of the improvement or show how latency behaves at longer contexts or higher concurrency.

Not covered here: other vLLM versions, other GPUs, CUDA graphs (`enforce_eager=False`), tensor parallelism, long-context throughput, and model quality.
